# Duplicated DLS Excitatory sessions

DANDI:000971 has 137 files with optogenetic treatment "Unknown". They come from the date-organized MED-PC files in `Opto Experiments/DLS Excitatory/`, which also hold runs that are already converted from the mice's own treatment folders. This notebook lists the sessions the fixed code converts and checks, for each published "Unknown" file, whether that run is still converted as "Unknown" or now only from the mouse's treatment folder. It then compares the events of each published duplicate pair.

In [1]:
from pathlib import Path

import h5py
import numpy as np
import pandas as pd

from lerner_lab_to_nwb.seiler_2024.seiler_2024_convert_dataset import opto_to_nwb, get_subject_session_start

data_dir_path = Path("/Volumes/T7/CatalystNeuro/Lerner/_Raw_Data_for_future_reference")
published_dir_path = Path("/Volumes/T7/CatalystNeuro/Lerner/000971")

In [2]:
session_to_nwb_kwargs_per_session = opto_to_nwb(
    data_dir_path=data_dir_path,
    output_dir_path=Path("unused"),
    start_variable="Start Date",
    verbose=False,
)
subject_session_start_to_treatments = {}
for session_to_nwb_kwargs in session_to_nwb_kwargs_per_session:
    subject_id, start_date, start_time = get_subject_session_start(session_to_nwb_kwargs)
    start = pd.Timestamp(f"{start_date} {start_time}").isoformat().replace(":", "-")
    subject_session_start_to_treatments.setdefault((subject_id, start), []).append(
        session_to_nwb_kwargs["optogenetic_treatment"]
    )

In [3]:
def parse_file_name(file_path: Path) -> tuple[str, str, str]:
    """Return the subject ID, treatment and start (YYYY-MM-DDTHH-MM-SS) of a published opto file."""
    subject_part, session_part = file_path.stem.split("_ses-")
    subject_id = subject_part.removeprefix("sub-").replace("-", ".")
    session_id = session_part.split("_")[0]
    treatment = session_id.split("-")[3]
    start = session_id[-19:]
    return subject_id, treatment, start


rows = []
for unknown_file_path in sorted(published_dir_path.glob("sub-*/*Opto-DLS-Excitatory-Unknown-*.nwb")):
    subject_id, _, start = parse_file_name(unknown_file_path)
    rows.append(
        dict(
            subject_id=subject_id,
            start=start,
            fixed_code_treatments=", ".join(subject_session_start_to_treatments[(subject_id, start)]),
        )
    )
unknown_df = pd.DataFrame(rows)
print(f"{len(unknown_df)} published Unknown files")
unknown_df["fixed_code_treatments"].value_counts()

137 published Unknown files


fixed_code_treatments
ChR2Scrambled    82
ChR2             53
Unknown           2
Name: count, dtype: int64

The runs that still convert as "Unknown" exist only in the date-organized files:

In [4]:
unknown_df[unknown_df["fixed_code_treatments"] == "Unknown"]

,subject_id,start,fixed_code_treatments
23,291.407,2020-09-23T12-41-13,Unknown
24,291.407,2020-09-24T12-22-41,Unknown


For the runs that are now converted only once, compare the events in the two published copies.

In [5]:
def read_behavior_datasets(file_path: Path) -> dict[str, np.ndarray]:
    datasets = {}
    with h5py.File(file_path, "r") as file:
        file["processing/behavior"].visititems(
            lambda name, item: datasets.__setitem__(name, item[()]) if isinstance(item, h5py.Dataset) else None
        )
    return datasets


comparison_rows = []
for row in unknown_df[unknown_df["fixed_code_treatments"] != "Unknown"].itertuples():
    subject_folder_path = published_dir_path / f"sub-{row.subject_id.replace('.', '-')}"
    (unknown_file_path,) = subject_folder_path.glob(f"*-Unknown-{row.start}*.nwb")
    (named_file_path,) = [
        file_path for file_path in subject_folder_path.glob(f"*{row.start}*.nwb") if file_path != unknown_file_path
    ]
    unknown_datasets = read_behavior_datasets(unknown_file_path)
    named_datasets = read_behavior_datasets(named_file_path)
    shared_names = sorted(set(unknown_datasets) & set(named_datasets))
    comparison_rows.append(
        dict(
            subject_id=row.subject_id,
            start=row.start,
            named_treatment=parse_file_name(named_file_path)[1],
            shared_datasets_identical=all(
                np.array_equal(unknown_datasets[name], named_datasets[name]) for name in shared_names
            ),
            only_in_unknown=", ".join(sorted(set(unknown_datasets) - set(named_datasets))),
            only_in_named=", ".join(sorted(set(named_datasets) - set(unknown_datasets))),
        )
    )
comparison_df = pd.DataFrame(comparison_rows)
comparison_df.groupby(["named_treatment", "shared_datasets_identical", "only_in_unknown", "only_in_named"]).size()

named_treatment  shared_datasets_identical  only_in_unknown             only_in_named
ChR2             True                                                                    48
                                            footshock_times/timestamps                    5
ChR2Scrambled    True                                                                    74
                                            footshock_times/timestamps                    8
dtype: int64

Every duplicate pair has identical events in the datasets both files hold. In 13 pairs the "Unknown" copy also has `footshock_times`, because the named copy was converted from a CSV export and the CSV reader drops the shock column. That is fixed separately (see #43).